In [ ]:
!wget https://raw.githubusercontent.com/meherafrozantar/PCA_Based_Gene_Selection/refs/heads/main/GSE6613_series_matrix.txt

!gunzip GSE6613_series_matrix.txt.gz



--2026-05-08 13:46:02--  https://raw.githubusercontent.com/meherafrozantar/PCA_Based_Gene_Selection/refs/heads/main/GSE6613_series_matrix.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11959101 (11M) [text/plain]
Saving to: ‘GSE6613_series_matrix.txt.1’

GSE6613_series_matr 100%[===================>]  11.40M  --.-KB/s    in 0.1s    

2026-05-08 13:46:02 (116 MB/s) - ‘GSE6613_series_matrix.txt.1’ saved [11959101/11959101]

gzip: GSE6613_series_matrix.txt.gz: No such file or directory


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. SET the PATH
data_path = '/content/'
matrix_file = os.path.join(data_path, 'GSE6613_series_matrix.txt')

# 2. LOAD DATA
# comment='!' skips the background info rows
df = pd.read_csv(matrix_file, sep='\t', comment='!', index_col=0)

# 3. PREPROCESS
# Transpose (Samples as rows), remove zero-variance genes, and Scale
df_pca = df.T
df_pca = df_pca.loc[:, df_pca.var() > 0]
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_pca)

# 4. RUN PCA (Variance = 0.95)
pca_95 = PCA(n_components=0.95)
pca_95.fit(scaled_data)

print("Step 1 & 2 COMPLETE!")
print(f"Matrix shape: {scaled_data.shape}")
print(f"Number of PCs needed for 95% variance: {pca_95.n_components_}")


Step 1 & 2 COMPLETE!
Matrix shape: (105, 22283)
Number of PCs needed for 95% variance: 91


## **Task 1: Run Principal Component Analysis (95% Variance).**
 In this step, PCA was applied to the standardized gene expression data from GSE6613. n_components parameter was set to 0.95, which tells the      algorithm to select the minimum number of Principal Components required to capture 95% of the total biological and technical variance in the dataset. This reduces the dimensionality of the data while preserving most of its information.

In [ ]:
def get_top_genes(pca_model, original_df, n_top_pcs):
    # 1. Get the loadings (coefficients) for the top n_top_pcs
    # Loadings are in pca_model.components_
    loadings = pca_model.components_[:n_top_pcs, :]

    # 2. Get the variance explained by each of these PCs
    var_explained = pca_model.explained_variance_ratio_[:n_top_pcs]

    # 3. Calculate Ultimate Weights: Sum(|Loading| * Variance_Explained)
    # This weights the importance of a gene by how much variance its PC explains
    absolute_loadings = np.abs(loadings)
    weighted_loadings = absolute_loadings.T * var_explained
    ultimate_weights = np.sum(weighted_loadings, axis=1)

    # 4. Map weights back to Gene Names
    gene_weights = pd.Series(ultimate_weights, index=original_df.columns)

    # 5. Sort and return top 20
    return gene_weights.sort_values(ascending=False).head(20)

#  Run Experiment 1: Top 5 PCAs
top_20_pca5 = get_top_genes(pca_95, df_pca, 5)
print("--- Top 20 Genes using Top 5 PCAs ---")
print(top_20_pca5)

# Run Experiment 2: Top 10 PCAs
top_20_pca10 = get_top_genes(pca_95, df_pca, 10)
print("\n--- Top 20 Genes using Top 10 PCAs ---")
print(top_20_pca10)


--- Top 20 Genes using Top 5 PCAs ---
ID_REF
200912_s_at             0.002783
204020_at               0.002721
AFFX-r2-Ec-bioD-3_at    0.002690
200627_at               0.002689
200063_s_at             0.002673
201051_at               0.002657
221199_at               0.002645
200858_s_at             0.002614
211699_x_at             0.002611
217232_x_at             0.002607
209116_x_at             0.002606
AFFX-r2-Ec-bioD-5_at    0.002605
211745_x_at             0.002603
209057_x_at             0.002603
211716_x_at             0.002600
AFFX-r2-Ec-bioC-5_at    0.002598
222012_at               0.002592
209458_x_at             0.002591
AFFX-BioDn-5_at         0.002589
AFFX-BioC-3_at          0.002584
dtype: float64

--- Top 20 Genes using Top 10 PCAs ---
ID_REF
211165_x_at        0.003287
221199_at          0.003282
215733_x_at        0.003234
216410_at          0.003232
210165_at          0.003224
206428_s_at        0.003221
211448_s_at        0.003218
211915_s_at        0.003215
219652_s_

# **Task 2-6: Gene Selection Algorithm**
## **Weighted PCA Gene Selection Process**

To identify the top genes, these steps were followed:
* **Extract Loadings:** Obtained the coefficients for each gene from the PCA model.
* **Absolute Values:** Converted coefficients to absolute values to measure the total impact of each gene.
* **Variance Weighting:** Multiplied the absolute values by the variance explained by each PC.
* **Ultimate Weights:** Summed these values to get a final score for each gene.
* **Ranking:** Sorted the genes to provide the **Top 20** list.


# **Task 7: Comparison and Final Commentary**

### **1. Is there any difference between the two lists?**
* **Yes.** Some high-variance genes appear in both lists, but the rankings and inclusion of certain genes differ between the **Top 5 PCA** and **Top 10 PCA** results.

### **2. Why is this happening?**
* This happens because each **Principal Component (PC)** captures a different "layer" of information.
* The **first 5 PCs** represent the most dominant signals in the dataset.
* Adding **PCs 6 through 10** introduces more subtle variations. Since our formula adds the coefficients of these extra PCs to the **"Ultimate Weight,"** genes that are highly dependent on these later components get higher scores and can move up into the Top 20 list.

### **3. Suitability of this approach for gene selection**
* **Strengths:** It is an **unsupervised method** that identifies important genes based solely on the data's mathematical structure, without needing to know if a sample is a patient or a control.
* **Weaknesses:** High variance (measured by PCA) does not always correspond to biological importance. A gene might vary due to technical noise rather than disease.
* **Verdict:** This approach is excellent for **exploratory data analysis** and reducing thousands of genes to a small set, but it should be confirmed with clinical tests for final results.
